<a href="https://colab.research.google.com/github/diazcas2/mIArmario/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''import sys
!pip install google-search-results --quiet
!pip install google-genai --quiet

# Instala numpy y scipy a versiones compatibles para evitar el ImportError de '_center'.
# Esto debe hacerse antes de transformers que tiene scipy como dependencia.
# Se utiliza --force-reinstall para asegurar que estas versiones sean las que se usan.
!pip install numpy==1.26.0 scipy==1.11.4 --force-reinstall --quiet

# Instala Pillow a una versión que satisfaga rembg (>=12.1.0) y sea compatible con otras librerías.
# Pillow 9.5.0 es compatible con torchvision y debería funcionar con las demás librerías.
!pip install Pillow==9.5.0

# Ahora instala las librerías principales. Deberían respetar la versión de Pillow ya instalada.
!pip install torch transformers google-generativeai langdetect --quiet
!pip install torchvision --quiet # Instala torchvision explícitamente
!pip install huggingface_hub
!pip install -q gradio_client

!pip install qwen-vl-utils -q

!pip install "rembg[cpu]" onnxruntime -q
!apt-get install -y fonts-dejavu -q

!pip install beautifulsoup4 -q

# Reiniciar el entorno de ejecución (Runtime) después de este cambio y ejecutar todo de nuevo.'''
# 1. Primero NumPy fijado
!pip install "numpy==1.26.4" --quiet

# 2. PyTorch y visión
!pip install torch torchvision --quiet

# 3. Transformers y Hugging Face
!pip install transformers huggingface_hub qwen-vl-utils --quiet

# 4. rembg (traerá Pillow>=12 automáticamente)
!pip install "rembg[cpu]" onnxruntime --quiet

# 5. Google GenAI (solo el nuevo)
!pip install google-genai --quiet

# 6. Resto de utilidades
!pip install google-search-results langdetect beautifulsoup4 gradio_client --quiet

# 7. Fuentes
!apt-get install -y fonts-dejavu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarra

In [2]:
import json
import re
import time
import torch
import google.generativeai as genai
from transformers import pipeline
from google.colab import userdata
from serpapi import GoogleSearch
import requests
import os

from transformers import WhisperProcessor, WhisperForConditionalGeneration
from IPython.display import display, Image as IPImage


from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/MIARMARIO/')

from Modulo1 import *
from Modulo30 import *
from Modulo2 import *
from Modulo40 import *


Mounted at /content/drive


In [3]:
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
whisper = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3",torch_dtype=torch.float32 )
device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = whisper.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [4]:
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')   # Google Cloud → Custom Search API
SEARCH_ENGINE_ID = userdata.get('SEARCH_ENGINE_ID') # programmablesearchengine.google.com → cx

SERPAPI_KEY = userdata.get('SERPAPI_KEY')  # serpapi.com → dashboard → API Key

CARPETA_ROPA      = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO"          # Carpeta con fotos del armario
RUTA_ARMARIO_JSON = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/armario.json"  # BD persistente de prendas

CARPETA_SALIDA    = '/content/drive/MyDrive/Colab Notebooks/MIARMARIO/CARPETA SALIDA'

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

# Paso 1. Cargar armario.

In [5]:
'''if forzar_reconstruccion and os.path.exists(RUTA_ARMARIO_JSON):
        os.remove(RUTA_ARMARIO_JSON)
        print("[Paso 1] Armario eliminado para reconstrucción completa")'''

if os.path.exists(RUTA_ARMARIO_JSON):
    print("[Paso 1] Usando armario.json existente.")
    armario = cargar_armario(RUTA_ARMARIO_JSON)
elif os.path.exists(CARPETA_ROPA):
    print("[Paso 1] Construyendo armario desde las fotos...")
    armario = modulo2_construir_armario(
        carpeta_ropa=CARPETA_ROPA,
        ruta_armario_json=RUTA_ARMARIO_JSON,
    )
else:
    raise FileNotFoundError("No hay armario.json ni carpeta ARMARIO. Añade fotos para continuar.")


[Paso 1] Usando armario.json existente.


In [6]:
print(json.dumps(armario, ensure_ascii=False, indent=2))

[
  {
    "tipo": "chaqueta",
    "color": "azul",
    "formalidad": "casual",
    "temporada": [
      "invierno"
    ],
    "id": "prenda_001_1",
    "imagen_path": "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/abrigo.jpg"
  },
  {
    "tipo": "pantalón",
    "color": "negro",
    "formalidad": "casual",
    "temporada": [
      "invierno",
      "primavera"
    ],
    "id": "prenda_002_1",
    "imagen_path": "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/pantalontraje.png"
  },
  {
    "tipo": "camisa",
    "color": "gris",
    "estampado": "liso",
    "formalidad": "formal",
    "temporada": [
      "primavera",
      "verano"
    ],
    "id": "prenda_003_1",
    "imagen_path": "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/polo.webp"
  },
  {
    "tipo": "camisa",
    "color": "azul",
    "estampado": "liso",
    "formalidad": "formal",
    "temporada": [
      "primavera",
      "verano"
    ],
    "id": "prenda_004_1",
    "imagen_path": "/conte

# Paso 2. Procesar instrucciones del ususario.


In [11]:
resultado_mod1 = modulo1_entrada(
    fuente="texto",
    texto="hola quiero que me digas un outfit que sea bueno para dar un paseo por la noche",
    ciudad="Yakutsk",
    model=gemini_model
)



In [ ]:
#a mano por si no quedan consultas
resultado_mod1 = {
  "texto": "Quiero un outfit para una cena esta noche. Mi jefe viene, así que necesito ir arreglado.",
  "fuente": "texto",
  "idioma": "es",
  "contexto": {
    "ciudad": "Sevilla",
    "temperatura": 19.2,
    "unidad": "celsius"
  }
}


In [ ]:
print(json.dumps(resultado_mod1, ensure_ascii=False, indent=2))

In [ ]:
# Ejemplo 2 — Entrada por voz
resultado_mod1 = modulo1_entrada(
    fuente="voz",
    ruta_audio="/content/drive/MyDrive/Colab Notebooks/MIARMARIO/mi_audio.mp4",# <-- cambia por tu archivo
    ciudad="Sevilla",
    model=model,
    whisper=whisper,
    processor=processor,
    device=device
)



In [12]:
print(json.dumps(resultado_mod1, ensure_ascii=False, indent=2))

{
  "texto": "Outfit para un paseo nocturno.",
  "fuente": "texto",
  "idioma": "es",
  "contexto": {
    "ciudad": "Yakutsk",
    "temperatura": -3.3,
    "unidad": "celsius"
  }
}


# Paso 3. Seleccionar outfit.

In [14]:
'''armario = {
    "armario": [
        {
            "id": "prenda_001",
            "tipo": "camisa",
            "color": "blanco",
            "estampado": "liso",
            "material": "algodón",
            "formalidad": "formal",
            "temporada": ["primavera", "verano", "otoño"],
            "imagen_path": "fotos/camisa_blanca_001.jpg"
        },
        {
            "id": "prenda_002",
            "tipo": "pantalón",
            "color": "negro",
            "estampado": "liso",
            "material": "lana",
            "formalidad": "formal",
            "temporada": ["otoño", "invierno"],
            "imagen_path": "fotos/pantalon_negro_002.jpg"
        }
    ]
}'''


resultado_mod3 = modulo3_seleccion_outfit(
    armario_json=armario,
    instruccion_usuario=resultado_mod1["texto"],
    contexto=resultado_mod1["contexto"],
    model=gemini_model,
    SERPAPI_KEY=SERPAPI_KEY
)



[Módulo 3] Outfits posibles detectados: 3
[Módulo 3] Queries generadas: ['americana negra casual hombre Zara', 'zapatos negros vestir hombre Mango', 'cinturón cuero negro hombre Zalando']
[Módulo 3] Resultados de tiendas encontrados: 8
[Módulo 3] Éxito: 3 outfits generados.


In [15]:
print(json.dumps(resultado_mod3, ensure_ascii=False, indent=2))

{
  "outfits": [
    {
      "nombre": "Noche Fría Urbana",
      "prendas": [
        {
          "id": "prenda_027_1",
          "tipo": "chaqueta",
          "color": "negro"
        },
        {
          "id": "prenda_029_1",
          "tipo": "sudadera",
          "color": "negro"
        },
        {
          "id": "prenda_014_1",
          "tipo": "pantalón",
          "color": "negro"
        },
        {
          "id": "prenda_013_1",
          "tipo": "zapatos",
          "color": "marrón"
        },
        {
          "id": "prenda_020_1",
          "tipo": "Bufanda",
          "color": "gris"
        },
        {
          "id": "prenda_021_1",
          "tipo": "reloj",
          "color": "negro"
        }
      ],
      "estilo": "casual",
      "ocasion": "Paseo nocturno",
      "temperatura": -3.3,
      "prompt_imagen": "A man wearing a black heavy winter coat, a black logo sweatshirt, dark black jeans, and brown casual sneakers. He is also wearing a grey scarf and

In [ ]:
#resultado por si no quedan consultas

resultado_mod3 = {
  "outfits": [
    {
      "nombre": "Elegancia Clásica de Noche",
      "prendas": [
        {
          "id": "prenda_002_1",
          "tipo": "pantalón",
          "color": "negro"
        },
        {
          "id": "prenda_004_1",
          "tipo": "camisa",
          "color": "azul"
        },
        {
          "id": "blazer_negro_tienda",
          "tipo": "chaqueta",
          "color": "negro"
        },
        {
          "id": "prenda_028_1",
          "tipo": "zapatos",
          "color": "negro"
        },
        {
          "id": "prenda_021_1",
          "tipo": "reloj",
          "color": "negro"
        }
      ],
      "estilo": "formal",
      "ocasion": "cena con el jefe",
      "temperatura": 19.2,
      "prompt_imagen": "A sophisticated man wearing a formal blue shirt, black dress pants, a black blazer, black formal loafers, and a black watch. Full body, fashion photography, studio lighting, looking confident and well-dressed.",
      "referencias_tiendas": [
        {
          "prenda": "BLAZER TRAJE CONFORT - Negro",
          "tienda": "https://www.zara.com › HOMBRE › COLECCIÓN",
          "enlace": "https://www.zara.com/es/es/hombre-blazers-l608.html",
          "imagen": "",
          "por_que": "Un blazer es esencial para un look formal y arreglado para una cena de negocios."
        }
      ]
    },
    {
      "nombre": "Sofisticación en Tonos Grises y Marrones",
      "prendas": [
        {
          "id": "prenda_023_1",
          "tipo": "pantalón",
          "color": "gris"
        },
        {
          "id": "prenda_003_1",
          "tipo": "camisa",
          "color": "gris"
        },
        {
          "id": "blazer_negro_tienda",
          "tipo": "chaqueta",
          "color": "negro"
        },
        {
          "id": "prenda_008_1",
          "tipo": "zapatos",
          "color": "marrón"
        },
        {
          "id": "prenda_019_1",
          "tipo": "cinturón",
          "color": "marrón oscuro"
        },
        {
          "id": "prenda_021_1",
          "tipo": "reloj",
          "color": "negro"
        }
      ],
      "estilo": "smart casual",
      "ocasion": "cena con el jefe",
      "temperatura": 19.2,
      "prompt_imagen": "A stylish man wearing a formal grey shirt, light grey chino pants, a black blazer, brown formal moccasins, a dark brown belt, and a black watch. Full body, fashion photography, studio lighting, looking confident and well-dressed.",
      "referencias_tiendas": [
        {
          "prenda": "BLAZER TRAJE CONFORT - Negro",
          "tienda": "https://www.zara.com › HOMBRE › COLECCIÓN",
          "enlace": "https://www.zara.com/es/es/hombre-blazers-l608.html",
          "imagen": "",
          "por_que": "Un blazer es esencial para un look formal y arreglado para una cena de negocios."
        }
      ]
    },
    {
      "nombre": "Toque Moderno y Cálido",
      "prendas": [
        {
          "id": "prenda_002_1",
          "tipo": "pantalón",
          "color": "negro"
        },
        {
          "id": "prenda_006_1",
          "tipo": "camiseta",
          "color": "gris"
        },
        {
          "id": "blazer_negro_tienda",
          "tipo": "chaqueta",
          "color": "negro"
        },
        {
          "id": "prenda_028_1",
          "tipo": "zapatos",
          "color": "negro"
        },
        {
          "id": "prenda_021_1",
          "tipo": "reloj",
          "color": "negro"
        }
      ],
      "estilo": "smart casual",
      "ocasion": "cena con el jefe",
      "temperatura": 19.2,
      "prompt_imagen": "A modern man wearing a grey fine knit sweater, black dress pants, a black blazer, black formal loafers, and a black watch. Full body, fashion photography, studio lighting, looking confident and well-dressed.",
      "referencias_tiendas": [
        {
          "prenda": "BLAZER TRAJE CONFORT - Negro",
          "tienda": "https://www.zara.com › HOMBRE › COLECCIÓN",
          "enlace": "https://www.zara.com/es/es/hombre-blazers-l608.html",
          "imagen": "",
          "por_que": "Un blazer es esencial para un look formal y arreglado para una cena de negocios."
        }
      ]
    },
    {
      "nombre": "Formal con Luminosidad",
      "prendas": [
        {
          "id": "prenda_023_1",
          "tipo": "pantalón",
          "color": "gris"
        },
        {
          "id": "prenda_004_1",
          "tipo": "camisa",
          "color": "azul"
        },
        {
          "id": "blazer_negro_tienda",
          "tipo": "chaqueta",
          "color": "negro"
        },
        {
          "id": "prenda_028_1",
          "tipo": "zapatos",
          "color": "negro"
        },
        {
          "id": "prenda_021_1",
          "tipo": "reloj",
          "color": "negro"
        }
      ],
      "estilo": "formal",
      "ocasion": "cena con el jefe",
      "temperatura": 19.2,
      "prompt_imagen": "A distinguished man wearing a formal blue shirt, light grey chino pants, a black blazer, black formal loafers, and a black watch. Full body, fashion photography, studio lighting, looking confident and well-dressed.",
      "referencias_tiendas": [
        {
          "prenda": "BLAZER TRAJE CONFORT - Negro",
          "tienda": "https://www.zara.com › HOMBRE › COLECCIÓN",
          "enlace": "https://www.zara.com/es/es/hombre-blazers-l608.html",
          "imagen": "",
          "por_que": "Un blazer es esencial para un look formal y arreglado para una cena de negocios."
        }
      ]
    },
    {
      "nombre": "Casual Elegante con Cuadros",
      "prendas": [
        {
          "id": "prenda_002_1",
          "tipo": "pantalón",
          "color": "negro"
        },
        {
          "id": "prenda_025_1",
          "tipo": "camisa",
          "color": "rojo/gris"
        },
        {
          "id": "blazer_negro_tienda",
          "tipo": "chaqueta",
          "color": "negro"
        },
        {
          "id": "prenda_028_1",
          "tipo": "zapatos",
          "color": "negro"
        },
        {
          "id": "prenda_021_1",
          "tipo": "reloj",
          "color": "negro"
        }
      ],
      "estilo": "smart casual",
      "ocasion": "cena con el jefe",
      "temperatura": 19.2,
      "prompt_imagen": "A fashion-forward man wearing a red and grey checkered informal shirt, black dress pants, a black blazer, black formal loafers, and a black watch. Full body, fashion photography, studio lighting, looking confident and well-dressed.",
      "referencias_tiendas": [
        {
          "prenda": "BLAZER TRAJE CONFORT - Negro",
          "tienda": "https://www.zara.com › HOMBRE › COLECCIÓN",
          "enlace": "https://www.zara.com/es/es/hombre-blazers-l608.html",
          "imagen": "",
          "por_que": "Un blazer es esencial para un look formal y arreglado para una cena de negocios."
        }
      ]
    }
  ]
}



print(json.dumps(resultado_mod3, ensure_ascii=False, indent=2))

In [ ]:
print("Referencias recibidas:", resultado_mod3.get("referencias_tiendas", []))
print("Cantidad:", len(resultado_mod3.get("referencias_tiendas", [])))

Referencias recibidas: []
Cantidad: 0


In [ ]:
ruta_jpg,ruta_lookbook = modulo4_generar_imagen(
    resultado_mod3 = resultado_mod3,
    armario        = armario,
    carpeta_salida = CARPETA_SALIDA,
    model          = gemini_model,
)

#display(IPImage(filename=ruta_lookbook))
print(f'Lookbook guardado en: {ruta_lookbook}')

[Módulo 4] Generando lookbook con 3 outfit(s)...


  0%|                                               | 0.00/176M [00:00<?, ?B/s]

# HTML

In [ ]:
"""
MIARMARIO — Servidor Flask para Google Colab
Pega este código en una celda de Colab y ejecútala.
Requiere que los módulos ya estén importados y los modelos cargados.
"""

!pip install flask flask-cors pyngrok -q

from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
import threading, time, os, tempfile, base64

app = Flask(__name__)
CORS(app)

# ─────────────────────────────────────────────
# CONFIGURA TU TOKEN AQUÍ
# ─────────────────────────────────────────────
NGROK_TOKEN   = userdata.get('TOKEN_NGROK')
CARPETA_SALIDA = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/lookbooks"
RUTA_ARMARIO  = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/armario.json"
CARPETA_ROPA  = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ropa"

os.makedirs(CARPETA_SALIDA, exist_ok=True)

# ─────────────────────────────────────────────
# ENDPOINTS
# ─────────────────────────────────────────────

@app.route("/ping", methods=["GET"])
def ping():
    return jsonify({"status": "ok"})


@app.route("/outfit", methods=["POST"])
def generar_outfit():
    try:
        instruccion = request.form.get("instruccion", "").strip()
        ciudad      = request.form.get("ciudad", "").strip() or None
        audio_file  = request.files.get("audio")
        fotos       = request.files.getlist("fotos")

        # ── 1. Guardar fotos nuevas en carpeta de ropa ──
        for foto in fotos:
            if foto and foto.filename:
                ruta = os.path.join(CARPETA_ROPA, foto.filename)
                foto.save(ruta)

        # ── 2. Módulo 1: procesar entrada ──
        if audio_file:
            # Guardar audio temporal y transcribir
            with tempfile.NamedTemporaryFile(suffix=".webm", delete=False) as tmp:
                audio_file.save(tmp.name)
                ruta_audio = tmp.name
            resultado_m1 = modulo1_entrada(
                fuente="voz",
                ruta_audio=ruta_audio,
                ciudad=ciudad,
                model=gemini_model,
                whisper=whisper_model,
                processor=whisper_processor,
                device=device
            )
            os.unlink(ruta_audio)
        elif instruccion:
            resultado_m1 = modulo1_entrada(
                fuente="texto",
                texto=instruccion,
                ciudad=ciudad,
                model=gemini_model
            )
        else:
            return jsonify({"error": "Se necesita texto o audio"}), 400

        # ── 3. Módulo 2: construir/actualizar armario ──
        if fotos:
            armario = modulo2_construir_armario(
                carpeta_ropa=CARPETA_ROPA,
                ruta_armario_json=RUTA_ARMARIO,
                model=qwen_model,
                processor=qwen_processor
            )
        else:
            armario = cargar_armario(RUTA_ARMARIO)

        # ── 4. Módulo 3: seleccionar outfit ──
        resultado_m3 = modulo3_seleccion_outfit(
            armario_json=armario,
            instruccion_usuario=resultado_m1["texto"],
            contexto=resultado_m1["contexto"],
            model=gemini_model,
            SERPAPI_KEY=SERPAPI_KEY
        )

        # ── 5. Módulo 4: generar lookbook ──
        ruta_jpg, ruta_html = modulo4_generar_imagen(
            resultado_mod3=resultado_m3,
            armario=armario,
            carpeta_salida=CARPETA_SALIDA,
            model=gemini_model
        )

        # ── 6. Leer HTML y devolverlo inline ──
        with open(ruta_html, "r", encoding="utf-8") as f:
            html_content = f.read()

        # Extraer metadata para el frontend
        outfits = resultado_m3.get("outfits", [resultado_m3.get("outfit", {})])
        primera = outfits[0] if outfits else {}
        ctx = resultado_m1.get("contexto", {})

        return jsonify({
            "html_content": html_content,
            "ocasion":      primera.get("ocasion", ""),
            "temperatura":  ctx.get("temperatura"),
            "ciudad":       ctx.get("ciudad", ciudad),
            "num_prendas":  sum(len(o.get("prendas", [])) for o in outfits),
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500


# ─────────────────────────────────────────────
# ARRANQUE
# ─────────────────────────────────────────────

def run_flask():
    app.run(port=5000, use_reloader=False, debug=False)

ngrok.set_auth_token(NGROK_TOKEN)
threading.Thread(target=run_flask, daemon=True).start()
time.sleep(2)

tunnel = ngrok.connect(5000)
print("─" * 50)
print(f"✅  URL pública: {tunnel.public_url}")
print(f"    Pégala en el HTML como URL del servidor")
print("─" * 50)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


──────────────────────────────────────────────────
✅  URL pública: https://inocencia-overtenacious-shavon.ngrok-free.dev
    Pégala en el HTML como URL del servidor
──────────────────────────────────────────────────
